In [ ]:
pip install -U langgraph langchain[openai] langchain-community e2b-code-interpreter

In [ ]:
pip install -U "langsmith[pytest]"

In [ ]:
!pip install -qU deepagents langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 187.6/187.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.4/50.4 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.6/67.6 kB 4.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 700.6/700.6 kB 17.2 MB/s eta 0:00:00


In [ ]:
!pip install deepagents tavily-python

In [ ]:
import os
from google.colab import userdata

os.environ['LANGSMITH_TRACING'] = 'true'
os.environ['LANGSMITH_ENDPOINT'] = 'https://eu.api.smith.langchain.com'
os.environ['LANGSMITH_API_KEY'] = userdata.get('LANGSMITH_API_KEY')
os.environ['LANGSMITH_PROJECT'] = 'pr-clear-muscle-85'
os.environ['OPENAI_API_KEY'] = userdata.get('OPENAI_API_KEY')

In [ ]:
os.environ['TAVILY_API_KEY'] = userdata.get('TAVILY_API_KEY')
os.environ['E2B_API_KEY'] = userdata.get('E2B_API_KEY')
os.environ['POLYGON_API_KEY'] = userdata.get('POLYGON_API_KEY')

In [ ]:
from deepagents import create_deep_agent

def get_weather(city: str) -> str:
    """Get weather for a given city."""
    return f"It's always sunny in {city}!"

agent = create_deep_agent(
    model="openai:gpt-5.4",
    tools=[get_weather],
    system_prompt="You are a helpful assistant",
)

# Run the agent
agent.invoke(
    {"messages": [{"role": "user", "content": "what is the weather in sf"}]}
)

{'messages': [HumanMessage(content='what is the weather in sf', additional_kwargs={}, response_metadata={}, id='b6082d4b-d0df-4ba7-ad7a-f01bc5d7a16e'),
  AIMessage(content=[{'arguments': '{"city":"San Francisco"}', 'call_id': 'call_jAEqq6M0Sh2OZAhAuxMw0tnD', 'name': 'get_weather', 'type': 'function_call', 'id': 'fc_0a1135e544da02360069fb583ead34819682de9d54c1381d73', 'status': 'completed'}], additional_kwargs={}, response_metadata={'id': 'resp_0a1135e544da02360069fb583da12481969c0f5f68570ff03f', 'created_at': 1778079805.0, 'metadata': {}, 'model': 'gpt-5.4-2026-03-05', 'object': 'response', 'service_tier': 'default', 'status': 'completed', 'model_provider': 'openai', 'model_name': 'gpt-5.4-2026-03-05'}, id='resp_0a1135e544da02360069fb583da12481969c0f5f68570ff03f', tool_calls=[{'name': 'get_weather', 'args': {'city': 'San Francisco'}, 'id': 'call_jAEqq6M0Sh2OZAhAuxMw0tnD', 'type': 'tool_call'}], invalid_tool_calls=[], usage_metadata={'input_tokens': 5201, 'output_tokens': 19, 'total_tok

In [ ]:
import os
from typing import Literal
from tavily import TavilyClient
from deepagents import create_deep_agent

tavily_client = TavilyClient(api_key=os.environ["TAVILY_API_KEY"])

def internet_search(
    query: str,
    max_results: int = 5,
    topic: Literal["general", "news", "finance"] = "general",
    include_raw_content: bool = False,
):
    """Run a web search"""
    return tavily_client.search(
        query,
        max_results=max_results,
        include_raw_content=include_raw_content,
        topic=topic,
    )

# System prompt to steer the agent to be an expert researcher
research_instructions = """You are an expert researcher. Your job is to conduct thorough research and then write a polished report.

You have access to an internet search tool as your primary means of gathering information.

## `internet_search`

Use this to run an internet search for a given query. You can specify the max number of results to return, the topic, and whether raw content should be included.
"""

In [ ]:
agent = create_deep_agent(
    model="openai:gpt-5.4",
    tools=[internet_search],
    system_prompt=research_instructions,
)

In [ ]:
result = agent.invoke({"messages": [{"role": "user", "content": "What is langgraph?"}]})

# Print the agent's response
print(result["messages"][-1].content)

[{'type': 'text', 'text': 'LangGraph is a framework for building stateful, multi-step AI agent workflows using graphs.\n\nIn practice, it lets you define:\n- nodes: steps in a workflow, often LLM calls or functions\n- edges: how execution moves between steps\n- state: shared data carried through the workflow\n- control flow: branching, looping, retries, and human-in-the-loop checkpoints\n\nIt’s commonly used for:\n- agent systems that need memory or persistence\n- tool-using workflows\n- multi-agent orchestration\n- long-running tasks with resumability\n- workflows that are too complex for a simple prompt chain\n\nIt’s closely associated with the LangChain ecosystem, but is designed to give lower-level control than basic chains.\n\nSimple mental model:\n- LangChain: higher-level abstractions for LLM apps\n- LangGraph: graph-based orchestration for complex agent behavior\n\nWhy people use it:\n- durable execution\n- explicit workflow structure\n- easier debugging of agent steps\n- suppo